# VisDrone Detection — Antlings Internship Assessment
Run all 5 tasks end-to-end on Kaggle T4 GPU.

In [ ]:
# Install dependencies
!pip install ultralytics lap -q
import ultralytics; ultralytics.checks()

In [ ]:
# Clone your repo (replace with your GitHub URL after pushing)
!git clone https://github.com/<your-username>/visdrone-detection.git
%cd visdrone-detection

In [ ]:
# Check dataset path (Kaggle input is at /kaggle/input/)
import os
os.listdir('/kaggle/input/')

## Task 1: Dataset Conversion

In [ ]:
# Update VISDRONE_ROOT in prepare_dataset.py to point to Kaggle input, or run inline:
import sys; sys.path.insert(0, '.')
from prepare_dataset import convert_split, visualize_samples
from pathlib import Path
import yaml

DATASET_PATH = '/kaggle/input/visdrone-dataset'  # adjust if needed

# Convert train split
convert_split(
    img_dir  = f'{DATASET_PATH}/VisDrone2019-DET-train/images',
    ann_dir  = f'{DATASET_PATH}/VisDrone2019-DET-train/annotations',
    out_img_dir = './visdrone_yolo/images/train',
    out_lbl_dir = './visdrone_yolo/labels/train',
)

# Convert val split
convert_split(
    img_dir  = f'{DATASET_PATH}/VisDrone2019-DET-val/images',
    ann_dir  = f'{DATASET_PATH}/VisDrone2019-DET-val/annotations',
    out_img_dir = './visdrone_yolo/images/val',
    out_lbl_dir = './visdrone_yolo/labels/val',
)

# Write data.yaml
yaml_content = {
    'path' : str(Path('./visdrone_yolo').resolve()),
    'train': 'images/train',
    'val'  : 'images/val',
    'nc'   : 2,
    'names': ['person', 'car'],
}
with open('./visdrone_yolo/data.yaml', 'w') as f:
    yaml.dump(yaml_content, f)

print('Dataset ready!')

In [ ]:
# Sample visualization
visualize_samples(
    Path('./visdrone_yolo/images/train'),
    Path('./visdrone_yolo/labels/train'),
    n=6
)

## Task 2: Training

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8m.pt')
results = model.train(
    data    = './visdrone_yolo/data.yaml',
    epochs  = 50,
    imgsz   = 1280,
    batch   = 8,
    patience= 15,
    mosaic  = 1.0,
    project = 'runs/visdrone',
    name    = 'yolov8m_exp',
    device  = 0,
)
print('Training done!')

## Task 3 & 4: Detection, Counting, and Tracking

In [ ]:
import subprocess

WEIGHTS = 'runs/visdrone/yolov8m_exp/weights/best.pt'
TEST_IMG = './visdrone_yolo/images/val'  # folder of val images

# Detection + counting on images
subprocess.run([
    'python', 'detect.py',
    '--source',  TEST_IMG,
    '--weights', WEIGHTS,
])

In [ ]:
# Show output images
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

imgs = list(Path('outputs/detections').glob('*.jpg'))[:6]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, img_path in zip(axes.flatten(), imgs):
    ax.imshow(mpimg.imread(str(img_path)))
    ax.set_title(img_path.name, fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Task 5: Evaluation

In [ ]:
subprocess.run([
    'python', 'evaluate.py',
    '--weights',   WEIGHTS,
    '--data',      './visdrone_yolo/data.yaml',
    '--fps-bench',
])